### Cronograma

| Etapa | Descrição | Resultado esperado |
|---|---|---|
| 0 | Definição do benchmark | Problema de referência, parâmetros e critérios de validação |
| 1 | Formulação matemática | Equações, hipóteses, domínio e condições de contorno |
| 2 | Discretização espacial | Construção manual dos operadores de diferenças finitas 2D |
| 3 | Construção do sistema linear | Montagem de \(A\eta=b\) e verificação da esparsidade |
| 4 | Análise matricial | Simetria, positividade definida, autovalores e condicionamento |
| 5 | Gradiente Conjugado | Implementação manual do CG |
| 6 | Precondicionamento | Implementação de Jacobi e IC(0) |
| 7 | Experimentos numéricos | Refinamento de malha e heterogeneidade dos coeficientes |
| 8 | Análise dos resultados | Comparações, gráficos, desempenho e conclusões numéricas |

# Etapa 0 - Definição  do Problema e Benchmark

De início, é necessário definir uma referência que permita testar e, idealmente, demonstrar que a implementação está correta. Vale ressaltar que isso é composto por dois objetivos complementares e diferentes: validar a formulação física e formular o algoritmo de solução do sistema linear.

Como referência principal será utilizada ["Semi-implicit finite difference methods for the two-dimensional shallow water equations" (Casulli, 1990)](https://doi.org/10.1016/0021-9991(90)90091-E), que estabelece uma abordagem matemática e algorítmica para simuladores hidrodinâmicos, abordando a resolução numérica das Equações de Águas Rasas Bidimensionais (2D Shallow Water Equations - SWE).

Esse sistema de equações descreve o escoamento de um fluido incompressível com superfície livre (como oceanos costeiros, estuários, grandes lagos, rios largos, mares continentais) sob a hipótese hidrostática (a pressão em qualquer ponto da coluna d'água é determinado exclusivamente pelo peso da coluna de fluido acima dele, ignorando os efeitos das acelerações verticais). Essa hipótese é válida quando a escala horizontal do escoamento (comprimento de onda) é muito maior que a profundidade vertical da coluna d'água,

$$\frac{H}{L} \ll 1$$

onde $H$ é a escala de profundidade vertical e $L$ a escala de comprimento horizontal característico (daí "águas rasas"). Em casos como, por exemplo, perto de pilastras de pontes, situações com forte turbulência vertical, convecção profunda com aceleração vertical do fluido, exigem o uso das equações de Navier-Stokes tridimensionais.

As SWE formam um sistema quasilenear hiperbólico de equações diferenciais parciais em três variáveis independentes $(x,y,t)$ ("quasilinear" porque é linear em relção às derivadas de ordem mais alta, mas os coeficientes dessas derivadas podem  depender das variáveis independentes e das próprias variáveis dependentes desconhecidas; "hiperbólico" porque o problema de valor inicial é bem posto, no sentido de o problema possuir solução única que depende continuamente dos dados iniciais e todas as perturbações iniciais se propagam a velocidades reais e finitas ao longo das curvas e superfícies características):

$$
    \boxed{
        \begin{gathered}
        \text{\small Equações de Águas Rasas} \\
        \hline \\[-8pt]
        \begin{aligned}
            &\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} + v\frac{\partial u}{\partial y} = -g\frac{\partial z}{\partial x} -\gamma u + \tau_x \\
            &\frac{\partial v}{\partial t} + u\frac{\partial v}{\partial x} + v\frac{\partial v}{\partial y} = -g\frac{\partial z}{\partial y} -\gamma v + \tau_y \\
            &\frac{\partial z}{\partial t} + \frac{\partial [(h+z)u]}{\partial x} + \frac{\partial [(h+z)v]}{\partial y} = 0
        \end{aligned}
        \end{gathered}
    }
$$

onde:

* $u(x,y,t)$ e $v(x,y,t)$ são as componentes da velocidade do fluido mediadas na profundidade, respectivamente nas direções $x$ e $y$;
* $z(x,y,t)$ é a elevação da superfície livre da água medida a partir de um nível de referência não perturbado;
* $h(x,y)$ é a profundidade da água também medida a partir de um nível de referência não perturbado;
* $g$, constante de aceleração da gravidade;
* $\tau_x$ e $\tau_y$ são os termos de estresse do vento nas direções $x$ e $y$, respectivamente;
* $\gamma$ é o coeficiente de atrito com o fundo, dado pela equação,

$$\gamma = \frac{g\sqrt{u^2 + v^2}}{C_z^2H}$$

sendo, $H(x,y,t) = h(x,y) + z(x,y,t)$ a profundidade total da água e $C_z$ o coeficiente de fricção de Chezy, que mede a capacidade de resistência ao escoamento imposta pelo leito sobre a velocidade média na vertical $(u,v)$ e a profundidade total $H$.

<figure align="center">
  <figcaption><strong>Figura 1: Domínio Físico, Escalas e Variáveis das Equações de Águas Rasas Bidimensionais (2D SWE)</strong></figcaption>
  <img src="../figures/swe_physical_domain.svg?v=4" alt="Domínio Físico, Escalas e Variáveis das Equações de Águas Rasas Bidimensionais (2D SWE)" width="100%"/>
  <figcaption><small>Fonte: Produzido pelo Autor.</small></figcaption>
</figure>

#### 0.1 Significado das equações

Na primeira equação,

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} + v\frac{\partial u}{\partial y} = -g\frac{\partial z}{\partial x} -\gamma u + \tau_x$$

no lado esquerdo, temos um conjunto de derivadas parciais que medem a mudança da velocidade $u$ no tempo $\frac{\partial u}{\partial t}$ e no campo espacial $x \times y$ onde $u$ varia: $\frac{\partial u}{\partial x}$ e $\frac{\partial u}{\partial y}$. No lado direito, sobre o termo $-g\frac{\partial z}{\partial x}$, como, pelo princípio hidrostático a pressão varia unicamente em função da coluna de água, a pressão horizontal depende da inclinação da superfície livre $z(x,y,t)$:

* Se a superfície da água está mais alta à esquerda e mais baixa a direita, então $\frac{\partial z}{\partial x} < 0$ e $-g\frac{\partial z}{\partial x} > 0$: a gravidade desloca a água da posição mais alta para a mais baixa, acelerando $u$ para a direita.
* Este termo impulsiona ou freia conforme o sinal da inclinação $\frac{\partial z}{\partial x}$.

Ainda na direita, o termo $-\gamma u$ atua como dissipação/amortecimento por atrito, opondo-se ao movimento, dado $\gamma \ge 0$. Por fim, $\tau_x$ um forçamento externo pelo vento na superfície, podendo acelerar ou desacelerar conforme a direção do vento.

<figure align="center">
  <figcaption><strong>Figura 2: Significado Físico e Dinâmico da Equação de Quantidade de Movimento na Direção x</strong></figcaption>
  <img src="../figures/swe_momentum_x_meaning.svg?v=4" alt="Significado Físico e Dinâmico da Equação de Quantidade de Movimento na Direção x" width="100%"/>
  <figcaption><small>Fonte: Produzido pelo Autor.</small></figcaption>
</figure>

Na segunda equação,

$$\frac{\partial v}{\partial t} + u\frac{\partial v}{\partial x} + v\frac{\partial v}{\partial y} = -g\frac{\partial z}{\partial y} -\gamma v + \tau_y$$

a lógica é exatamente a mesma, mas para a componente $v$ na horizontal $y$.

Na terceira equação,

$$\frac{\partial z}{\partial t} + \frac{\partial [(h+z)u]}{\partial x} + \frac{\partial [(h+z)v]}{\partial y} = 0$$

descreve o sobe-e-desce $\frac{\partial z}{\partial t}$ com conservação de massa para um fluido incomprensível, ou seja, sem evaporação, sem chuva ou infiltração, daí ser igual a $0$. Podemos reescrevê-la como

$$\frac{\partial x}{\partial t} = -\left(\frac{\partial [(h + z)u]}{\partial x} + \frac{\partial [(h + z)v]}{\partial y}\right)$$

onde $(h + z)x$ e $(h + z)v$ são os fluxos volumétricos por unidade de largura nas direções $x$ e $y$, ou seja, a profundidade total vezes a velocidade. Se entra mais água do que sai na coluna d'água, o divergente do fluxo é negativo, o lado direito torna-se positivo e $frac{\partial z}{\partial t} > 0$, logo a coluna de água sobre; se o gradiante for positivo, a coluna de água desce.

<figure align="center">
  <figcaption><strong>Figura 3: Significado Físico da Equação da Continuidade (Conservação de Massa e Sobe-e-Desce da Superfície Livre)</strong></figcaption>
  <img src="../figures/swe_continuity_meaning.svg?v=2" alt="Significado Físico da Equação da Continuidade (Conservação de Massa e Sobe-e-Desce da Superfície Livre)" width="100%"/>
  <figcaption><small>Fonte: Produzido pelo Autor.</small></figcaption>
</figure>

# Etapa 1 - Formulação Matemática

# Etapa 2 - Discretização Espacial

# Etapa 3 - Construção do Sistema Linear

# Etapa 4 - Análise Matricial

# Etapa 5 - Gradiente Conjugado

# Etapa 6 - Precondicionamento

# Etapa 7 - Experimentos Numéricos

# Etapa 8 - Análise dos Resultados